# EPIC-KITCHENS-100 Multi-Instance Retrieval – Exploratory Data Analysis (subset)

This notebook performs a lightweight exploratory analysis of the EPIC-KITCHENS-100 dataset focusing on the Multi-Instance Retrieval (MIR) split. It uses only annotation CSV files and an optional small subset of RGB frames, following the official repository structure so that it runs quickly once the annotations are available locally.

## 1. Setup and data loading

We first import the required Python packages and load the main metadata files:

- `EPIC_100_video_info.csv` with per-video duration, frame rate and resolution.
- `retrieval_annotations/EPIC_100_retrieval_train.csv` with MIR training segments.

If the CSV files exist locally under `data/epic-kitchens-100-annotations/`, they are loaded from disk; otherwise, they are fetched from the official GitHub repository via raw URLs (small enough to download quickly).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set(style="whitegrid")

base_annotations = Path("data/epic-kitchens-100-annotations")


def load_csv(rel_path: str, raw_url: str) -> pd.DataFrame:
    local_path = base_annotations / rel_path
    if local_path.exists():
        print(f"Loading local file: {local_path}")
        return pd.read_csv(local_path)
    else:
        print(f"Local file not found, downloading from {raw_url}")
        return pd.read_csv(raw_url)


video_info = load_csv(
    "EPIC_100_video_info.csv",
    "https://raw.githubusercontent.com/epic-kitchens/epic-kitchens-100-annotations/master/EPIC_100_video_info.csv",
)

retrieval_train = load_csv(
    "retrieval_annotations/EPIC_100_retrieval_train.csv",
    "https://raw.githubusercontent.com/epic-kitchens/epic-kitchens-100-annotations/master/retrieval_annotations/EPIC_100_retrieval_train.csv",
)

video_info.head(), retrieval_train.head()

## 2. Video-level statistics (duration, FPS, resolution)

Using `EPIC_100_video_info.csv` we compute descriptive statistics of video duration, frame rate and spatial resolution, as well as the average number of pixels per frame. We also visualise the distributions to show that most videos share a common Full HD resolution and frame rate.

In [ ]:
# Parse resolution and compute number of pixels per frame
video_info[["width", "height"]] = video_info["resolution"].str.split("x", expand=True).astype(float)
video_info["n_pixels"] = video_info["width"] * video_info["height"]

print("Video duration (seconds):")
print(video_info["duration"].describe())
print("Video FPS:")
print(video_info["fps"].describe())
print("Pixels per frame:")
print(video_info["n_pixels"].describe())

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

sns.histplot(video_info["duration"], bins=40, ax=axes[0])
axes[0].set_title("Video duration distribution (s)")
axes[0].set_xlabel("Duration (seconds)")

sns.histplot(video_info["fps"], bins=20, ax=axes[1])
axes[1].set_title("Frame rate distribution (FPS)")
axes[1].set_xlabel("Frames per second")

sns.histplot(video_info["n_pixels"], bins=20, ax=axes[2])
axes[2].set_title("Pixels per frame distribution")
axes[2].set_xlabel("Number of pixels")

plt.tight_layout()

## 3. Resolution distribution

We next examine the distribution of spatial resolutions across videos to confirm that most clips are recorded at a common Full HD resolution, with only a few exceptional cases at different sizes inherited from earlier dataset versions.

In [ ]:
resolution_counts = video_info["resolution"].value_counts().reset_index()
resolution_counts.columns = ["resolution", "count"]

print(resolution_counts.head(10))

plt.figure(figsize=(8, 4))
sns.barplot(data=resolution_counts.head(10), x="resolution", y="count")
plt.title("Top video resolutions")
plt.xlabel("Resolution (WxH)")
plt.ylabel("Number of videos")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

## 4. Segment-level statistics for MIR

For the Multi-Instance Retrieval subset, we combine the retrieval training annotations with the video metadata to compute the duration of each action segment in seconds. This allows us to report the average clip length and to visualise the distribution of segment durations.

In [ ]:
# Merge FPS information into retrieval annotations
segments = retrieval_train.merge(video_info[["video_id", "fps"]], on="video_id", how="left")

# Duration in frames and seconds
segments["segment_frames"] = segments["stop_frame"] - segments["start_frame"] + 1
segments["segment_seconds"] = segments["segment_frames"] / segments["fps"]

print("Segment duration (seconds):")
print(segments["segment_seconds"].describe())

plt.figure(figsize=(8, 4))
sns.histplot(segments["segment_seconds"], bins=40)
plt.title("Segment duration distribution (MIR train)")
plt.xlabel("Duration (seconds)")
plt.tight_layout()

## 5. Example frame from a local subset

To illustrate the visual content, we display one RGB frame from a locally downloaded subset of EPIC-KITCHENS frames (obtained with the official download scripts). Update the `frame_path` below to point to any JPEG frame in your subset; if the file does not exist, the cell simply prints a message and skips the plot.

In [ ]:
from pathlib import Path
from PIL import Image

# TODO: update this path to a valid frame on your machine
frame_path = Path("rgb/P01_109/frame_0000000001.jpg")

if frame_path.exists():
    img = Image.open(frame_path)
    print(f"Loaded frame: {frame_path}")
    print(f"Image size (width x height): {img.size}")
    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Example EPIC-KITCHENS frame")
else:
    print(f"Example frame not found at {frame_path}. Update 'frame_path' to point to a local JPEG frame.")